# Análisis de métricas de UniHelp

Este cuaderno es el **único lugar** del proyecto donde se calcula una métrica. El backend produce trazas, este cuaderno las valida y calcula, y el panel web solo lee `salidas/resultados.json` (decisión 19).

- Se ejecuta de principio a fin sin intervención: `pnpm analisis` desde la raíz del repositorio.
- Cada tabla o figura publicable sale de una celda con la etiqueta de su archivo (`tabla_2`, `figura_1`…), visible en los metadatos de la celda.
- La unidad de observación es la ejecución; la unidad de **análisis** es la tarea. Todo intervalo remuestrea tareas y el tamaño de muestra que se cita es el número de tareas.
- Ejecutarlo dos veces sobre los mismos datos produce archivos idénticos salvo `generado_en`.

In [ ]:
# Parametros de papermill (-p nombre valor). Rutas relativas a experiment/.
directorio_corrida = 'fixtures/sinteticas'
directorio_salidas = 'salidas'
directorio_tareas = '../docs/tasks'
# ISO 8601 UTC. None = ahora. Es lo UNICO que cambia entre dos ejecuciones sobre los mismos datos.
generado_en = None

In [ ]:
import sys
from datetime import UTC, datetime
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f'El cuaderno exige el Python 3.12 del entorno uv de experiment/; se ejecuta con {sys.version}')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analisis.carga import consolidar, escribir_parquet, leer_parquet
from analisis.familias import Contexto, calcular
from analisis.familias.m1_efectividad import distribucion_exitos
from analisis.inferencia import ConfiguracionBootstrap
from analisis.registro import cargar_registro
from analisis.salida import Publicador, construir_resultados, filas_como_tabla

corrida = Path(directorio_corrida)
salidas = Path(directorio_salidas)
generado_en = generado_en or datetime.now(UTC).strftime('%Y-%m-%dT%H:%M:%SZ')
registro = cargar_registro()
print(f'Registro {registro.version}: {len(registro.metricas)} metricas, {len(registro.implementadas())} implementadas')

## 1. Carga y validación de trazas

Un intento rechazado nunca llega a `intermedios/ejecuciones.parquet`. Las métricas se calculan desde ese Parquet releído, no desde memoria.

In [ ]:
consolidado = consolidar(corrida, registro, Path(directorio_tareas))
escribir_parquet(consolidado, salidas / 'intermedios')
ejecuciones, intentos = leer_parquet(salidas / 'intermedios')
print(f'Intentos: {len(intentos)} · validos: {len(ejecuciones)} · rechazados: {len(consolidado.rechazos)}')
pd.DataFrame(
    [{'origen': r.origen, 'ejecucion': r.run_id, 'motivos': ', '.join(r.motivos)} for r in consolidado.rechazos],
    columns=['origen', 'ejecucion', 'motivos'],
)

In [ ]:
umbral_aviso = registro.validaciones['residuo_orquestacion']['aviso_proporcion']
print(f'Ejecuciones validas con residuo de orquestacion por encima de {umbral_aviso:.0%}:')
pd.DataFrame(
    [{'ejecucion': a.run_id, 'proporcion_residuo': round(a.proporcion, 4)} for a in consolidado.avisos],
    columns=['ejecucion', 'proporcion_residuo'],
)

## 2. Cálculo de métricas

Solo las métricas **primarias** sostienen conclusiones. Las de resultado abierto no llevan indicador de umbral; las de tipo umbral indican si lo alcanzan.

In [ ]:
configuracion = ConfiguracionBootstrap(
    replicas=registro.inferencia['replicas'],
    nivel=registro.inferencia['nivel'],
    semilla=registro.inferencia['semilla'],
)
contexto = Contexto(registro, ejecuciones, intentos, consolidado.insumos, configuracion, corrida)
resultados = calcular(contexto)
documento = construir_resultados(
    contexto, resultados, consolidado.rechazos, consolidado.avisos, generado_en=generado_en
)
pd.DataFrame(
    [
        {
            'codigo': m['codigo'],
            'rol': m['rol'],
            'tipo_valor_esperado': m['tipo_valor_esperado'],
            'estado': m['estado'],
            'alcanza_umbral': m['umbral']['alcanza'] if 'umbral' in m else '(no aplica)',
        }
        for m in documento['metricas']
    ]
)

## 3. Tablas y figuras publicables

Numeradas como las plantillas de reporte del plan de medición (sección 17). Las tablas 3, 5 y 6 dependen de familias aún no implementadas.

Paleta categórica validada (orden fijo, un color por arquitectura). Dos colores quedan por debajo de 3:1 de contraste con el fondo, así que cada figura tiene su tabla con los mismos valores.

In [ ]:
publicador = Publicador(salidas)
ARQUITECTURAS = list(registro.arquitecturas)
N_TAREAS = documento['corrida']['inferencia']['tamano_muestra']
COLOR_ARQUITECTURA = dict(zip(ARQUITECTURAS, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100'], strict=True))
COLOR_DESGLOSE = ['#e87ba4', '#008300', '#4a3aa7', '#e34948']
SUPERFICIE, TINTA, TINTA_SECUNDARIA, EJE, REJILLA = '#fcfcfb', '#0b0b0b', '#52514e', '#c3c2b7', '#e1e0d9'
ETIQUETAS = {
    'global': 'Global', 'informativa': 'Informativa', 'diagnostico': 'Diagnóstico',
    'compuesta': 'Compuesta', 'adversarial': 'Adversarial',
    'modelo': 'Modelo', 'herramienta': 'Herramienta', 'transporte': 'Transporte', 'orquestacion': 'Orquestación',
    'timeout': 'Tiempo agotado', 'limite_herramientas': 'Límite de herramientas',
    'error_agente': 'Error del agente', 'fallo_rubrica': 'Fallo de la rúbrica',
}
plt.rcParams.update({
    'figure.facecolor': SUPERFICIE, 'axes.facecolor': SUPERFICIE, 'savefig.facecolor': SUPERFICIE,
    'font.size': 9, 'axes.titlesize': 11, 'axes.titlelocation': 'left', 'axes.titlecolor': TINTA,
    'axes.edgecolor': EJE, 'axes.labelcolor': TINTA_SECUNDARIA, 'text.color': TINTA,
    'xtick.color': TINTA_SECUNDARIA, 'ytick.color': TINTA_SECUNDARIA,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': REJILLA, 'grid.linewidth': 0.6,
    'legend.frameon': False, 'svg.fonttype': 'path',
})


def tabla_de(codigo):
    resultado = resultados[codigo]
    return filas_como_tabla(resultado) if resultado.estado == 'calculada' else None


def tabla_sin_datos(*codigos):
    return pd.DataFrame(
        [{'codigo': c, 'estado': resultados[c].estado, 'motivo': resultados[c].motivo_estado} for c in codigos]
    )


def ordenar_columnas(tabla, primeras):
    return tabla[[c for c in primeras if c in tabla] + [c for c in tabla if c not in primeras]]


def figura_sin_datos(titulo, *codigos):
    figura, eje = plt.subplots(figsize=(8, 2))
    eje.axis('off')
    eje.set_title(titulo)
    eje.text(0, 0.5, 'Sin datos: ' + '; '.join(f'{c}: {resultados[c].motivo_estado}' for c in codigos), color=TINTA_SECUNDARIA)
    return figura

In [ ]:
# Tabla 1 · Piso de latencia por transporte (M4.3), en milisegundos.
piso = tabla_de('M4.3')
if piso is None:
    tabla_1 = tabla_sin_datos('M4.3')
else:
    tabla_1 = (
        piso.pivot(index='transporte', columns='estadistico', values='valor')
        .reindex(columns=['p50', 'p95', 'p99', 'desviacion_estandar'])
        .reset_index()
    )
publicador.tabla('tabla_1', tabla_1)
tabla_1

In [ ]:
# Tabla 2 · Efectividad por arquitectura y categoria con IC 95 % (M1.1, M1.2).
exito_global, exito_categoria = tabla_de('M1.1'), tabla_de('M1.2')
if exito_global is None or exito_categoria is None:
    tabla_2 = tabla_sin_datos('M1.1', 'M1.2')
else:
    tabla_2 = pd.concat([exito_global.assign(categoria='global'), exito_categoria], ignore_index=True)
    tabla_2['categoria'] = pd.Categorical(tabla_2['categoria'], ['global', *registro.categorias], ordered=True)
    tabla_2 = tabla_2.sort_values(['categoria', 'arquitectura'], kind='stable').reset_index(drop=True)
    tabla_2 = tabla_2[['categoria', 'arquitectura', 'valor', 'ic_inferior', 'ic_superior', 'n_tareas', 'n_observaciones']]
publicador.tabla('tabla_2', tabla_2)
tabla_2

In [ ]:
# Tabla 4 · Costo por arquitectura (M4.1, M4.4 a M4.7).
partes = []
for codigo in ('M4.1', 'M4.4', 'M4.5', 'M4.6', 'M4.7'):
    tabla = tabla_de(codigo)
    partes.append(tabla_sin_datos(codigo) if tabla is None else tabla.assign(codigo=codigo))
tabla_4 = ordenar_columnas(
    pd.concat(partes, ignore_index=True),
    ['codigo', 'arquitectura', 'tokens', 'estadistico', 'valor', 'ic_inferior', 'ic_superior', 'n_tareas', 'n_observaciones'],
)
publicador.tabla('tabla_4', tabla_4)
tabla_4

In [ ]:
# Tabla 7 · Fiabilidad del experimento (M7): valor observado, umbral y consecuencia.
tabla_7 = pd.DataFrame(
    [
        {
            'codigo': m['codigo'],
            'nombre': m['nombre'],
            'estado': m['estado'],
            'observado': m['umbral']['observado'],
            'umbral': m['umbral']['descripcion'],
            'alcanza': m['umbral']['alcanza'],
            'consecuencia': m['umbral'].get('consecuencia', ''),
            'motivo_estado': m['motivo_estado'],
        }
        for m in documento['metricas']
        if m['familia'] == 'M7'
    ]
)
publicador.tabla('tabla_7', tabla_7)
tabla_7

In [ ]:
# Figura 1 · Efectividad con intervalos (M1.1, M1.2). Valores en tabla_2.
titulo = f'Tasa de éxito con IC 95 % · n = {N_TAREAS} tareas'
if 'valor' not in tabla_2:
    figura = figura_sin_datos(titulo, 'M1.1', 'M1.2')
else:
    grupos = ['global', *registro.categorias]
    figura, eje = plt.subplots(figsize=(8, 4.2))
    for posicion, arquitectura in enumerate(ARQUITECTURAS):
        filas = tabla_2[tabla_2['arquitectura'] == arquitectura].set_index('categoria').reindex(grupos)
        x = np.arange(len(grupos)) + (posicion - 1.5) * 0.17
        y = filas['valor'].to_numpy(float)
        error = [y - filas['ic_inferior'].to_numpy(float), filas['ic_superior'].to_numpy(float) - y]
        eje.errorbar(x, y, yerr=error, fmt='o', markersize=6, elinewidth=2, capsize=0,
                     color=COLOR_ARQUITECTURA[arquitectura], label=arquitectura)
    eje.set_xticks(range(len(grupos)), [ETIQUETAS[g] for g in grupos])
    eje.set_ylim(0, 1.02)
    eje.set_ylabel('Tasa de éxito (media entre tareas)')
    eje.set_title(titulo, pad=26)
    eje.legend(ncols=len(ARQUITECTURAS), loc='lower left', bbox_to_anchor=(0, 1.0), borderaxespad=0.2)
    figura.text(0.01, 0.01, 'Por categoría hay pocas tareas: su intervalo es ancho y la diferencia puntual sola no se interpreta.',
                color=TINTA_SECUNDARIA, fontsize=8)
    figura.tight_layout(rect=(0, 0.04, 1, 1))
publicador.figura('figura_1', figura)
plt.show()

In [ ]:
# Figura 2 · Descomposicion de la latencia (M4.2) con la mediana de extremo a extremo (M4.1). Valores en tabla_4 y resultados.json.
titulo = 'Descomposición de la latencia por arquitectura'
descomposicion, extremo = tabla_de('M4.2'), tabla_de('M4.1')
if descomposicion is None or extremo is None:
    figura = figura_sin_datos(titulo, 'M4.2', 'M4.1')
else:
    componentes = ['modelo', 'herramienta', 'transporte', 'orquestacion']
    medianas = (
        descomposicion[descomposicion['estadistico'] == 'mediana_entre_tareas']
        .pivot(index='arquitectura', columns='componente', values='valor')
        .reindex(index=ARQUITECTURAS)
    )
    figura, eje = plt.subplots(figsize=(8, 4.4))
    apilado = np.zeros(len(ARQUITECTURAS))
    for componente, color in zip(componentes, COLOR_DESGLOSE, strict=True):
        valores = medianas[componente].fillna(0).to_numpy(float) / 1000
        eje.bar(ARQUITECTURAS, valores, bottom=apilado, width=0.55, color=color,
                edgecolor=SUPERFICIE, linewidth=2, label=ETIQUETAS[componente])
        apilado += valores
    total = extremo[extremo['estadistico'] == 'mediana_entre_tareas'].set_index('arquitectura').reindex(ARQUITECTURAS)['valor'].to_numpy(float) / 1000
    posiciones = np.arange(len(ARQUITECTURAS))
    eje.hlines(total, posiciones - 0.32, posiciones + 0.32, colors=TINTA, linewidth=2, zorder=3,
               label='Mediana de extremo a extremo')
    residuo = medianas['proporcion_orquestacion']
    for posicion, arquitectura in enumerate(ARQUITECTURAS):
        eje.annotate(f'{total[posicion]:.1f} s · residuo {residuo[arquitectura]:.1%}',
                     (posicion, max(apilado[posicion], total[posicion])), textcoords='offset points',
                     xytext=(0, 6), ha='center', color=TINTA_SECUNDARIA, fontsize=8)
    eje.set_ylabel('Segundos (mediana entre tareas)')
    eje.set_title(titulo, pad=44)
    eje.legend(ncols=3, loc='lower left', bbox_to_anchor=(0, 1.0), borderaxespad=0.2)
    figura.text(0.01, 0.01, 'Cada segmento es la mediana de su componente; las medianas no son aditivas.',
                color=TINTA_SECUNDARIA, fontsize=8)
    figura.tight_layout(rect=(0, 0.04, 1, 1))
publicador.figura('figura_2', figura)
plt.show()

In [ ]:
# Figura 3 · Estabilidad: repeticiones exitosas por tarea (M1.3, M1.4).
titulo = f'Repeticiones exitosas por tarea · {N_TAREAS} tareas'
distribucion = distribucion_exitos(registro.metrica('M1.3'), contexto)
if distribucion is None:
    figura = figura_sin_datos(titulo, 'M1.3')
else:
    maximo = int(distribucion['repeticiones'].max())
    figura, ejes = plt.subplots(1, len(ARQUITECTURAS), figsize=(9, 3.2), sharey=True)
    for eje, arquitectura in zip(ejes, ARQUITECTURAS, strict=True):
        exitos = distribucion.loc[distribucion['arquitectura'] == arquitectura, 'exitos'].astype(int)
        conteo = exitos.value_counts().reindex(range(maximo + 1), fill_value=0)
        eje.bar(conteo.index, conteo.to_numpy(), width=0.8, color=COLOR_ARQUITECTURA[arquitectura],
                edgecolor=SUPERFICIE, linewidth=2)
        eje.set_title(arquitectura)
        eje.set_xticks(range(maximo + 1))
        eje.set_xlabel('Repeticiones exitosas')
    ejes[0].set_ylabel('Tareas')
    figura.suptitle(titulo, x=0.01, ha='left', fontsize=11)
    figura.tight_layout()
publicador.figura('figura_3', figura)
plt.show()

In [ ]:
# Figura 4 · Fallos por tipo (M1.5). Valores en resultados.json.
titulo = 'Composición de los fallos por arquitectura'
fallos = tabla_de('M1.5')
if fallos is None:
    figura = figura_sin_datos(titulo, 'M1.5')
else:
    conteos = fallos[fallos['estadistico'] == 'conteo']
    tipos = list(dict.fromkeys(conteos['tipo_fallo']))
    por_tipo = conteos.pivot(index='arquitectura', columns='tipo_fallo', values='valor').reindex(index=ARQUITECTURAS, columns=tipos)
    figura, eje = plt.subplots(figsize=(8, 4.2))
    apilado = np.zeros(len(ARQUITECTURAS))
    for tipo, color in zip(tipos, COLOR_DESGLOSE, strict=True):
        valores = por_tipo[tipo].fillna(0).to_numpy(float)
        eje.bar(ARQUITECTURAS, valores, bottom=apilado, width=0.55, color=color,
                edgecolor=SUPERFICIE, linewidth=2, label=ETIQUETAS.get(tipo, tipo))
        apilado += valores
    for posicion, total in enumerate(apilado):
        eje.annotate(f'{total:.0f}', (posicion, total), textcoords='offset points', xytext=(0, 4),
                     ha='center', color=TINTA_SECUNDARIA, fontsize=8)
    eje.set_ylabel('Ejecuciones no exitosas')
    eje.set_title(titulo, pad=26)
    eje.legend(ncols=len(tipos), loc='lower left', bbox_to_anchor=(0, 1.0), borderaxespad=0.2)
    figura.tight_layout()
publicador.figura('figura_4', figura)
plt.show()

## 4. Contrato con el panel

`resultados.json` valida contra `schemas/resultados.schema.json` antes de escribirse. `manifiesto.json` lista cada salida con su etiqueta y su SHA-256.

In [ ]:
ruta_resultados = publicador.resultados(documento)
ruta_manifiesto = publicador.manifiesto(documento)
print(f'Resultados: {ruta_resultados}')
print(f'Manifiesto: {ruta_manifiesto}')
print(f"Generado en {documento['generado_en']} · tamano de muestra para inferencia: {N_TAREAS} tareas")